# Embedding & indexing playground
Run with the **`.venv-ingest`** kernel (GPU env). Sections 1-3 run fully offline. Section 4 is optional: it runs only if `POSTGRES_PASSWORD` (and optionally `POSTGRES_USER/HOST/PORT/DB`) is exported and Postgres is up; its session is server-enforced read-only, so nothing here can write to the live DB. This notebook never reads `.env`.

1. **Setup**: paths, DB connection, embedder
2. **Offline structure**: cached Docling blocks -> `build_structure` -> provisions/chunks (no DB)
3. **Embedding sandbox**: embed any text/node in memory, compare, rank
4. **Live index**: inspect stored provisions, chunks, vectors; vector search; print any node

## 1. Setup

In [ ]:
import asyncio, hashlib, json, math, os, sys, textwrap
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "app").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from sqlalchemy import text
from sqlalchemy.engine import URL
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
print("root:", ROOT, "| live DB configured:", bool(os.environ.get("POSTGRES_PASSWORD")))


In [ ]:
# Loads Harrier on GPU (falls back to Ollama nomic-embed-text). Takes a few seconds the first time.
from app.ingest.embedding import load_embedder

embedder = load_embedder("auto", ollama_url=OLLAMA_URL)
print(embedder.model, "| revision:", embedder.revision[:12], "| dims:", embedder.dimensions)

In [ ]:
# Shared helpers
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


def norm(v):
    return math.sqrt(sum(x * x for x in v))


def show_vector(v, head=8):
    print(f"dims={len(v)}  L2-norm={norm(v):.4f}  head={[round(x, 4) for x in v[:head]]}")


def wrap(s, width=110, limit=1500):
    s = s if len(s) <= limit else s[:limit] + f"... [+{len(s) - limit} chars]"
    return "\n".join(textwrap.fill(line, width) if line.strip() else "" for line in s.splitlines())


DB_OK = False
engine = sessions = None


async def check_db():
    """Optional live-index preflight. Env vars only (POSTGRES_PASSWORD required); never reads .env. Read-only session."""
    global DB_OK, engine, sessions
    password = os.environ.get("POSTGRES_PASSWORD")
    if not password:
        print("POSTGRES_PASSWORD not exported -> live-index cells skipped (sections 1-3 are unaffected).")
        return
    url = URL.create(
        "postgresql+asyncpg", os.environ.get("POSTGRES_USER", "civil_chatbot_owner"), password,
        os.environ.get("POSTGRES_HOST", "127.0.0.1"), int(os.environ.get("POSTGRES_PORT", "5432")),
        os.environ.get("POSTGRES_DB", "civil_chatbot"),
    )
    engine = create_async_engine(url, connect_args={"server_settings": {"default_transaction_read_only": "on"}})
    sessions = async_sessionmaker(engine, expire_on_commit=False)
    try:
        async with sessions() as session:
            await session.execute(text("SELECT 1"))
        DB_OK = True
        print("Postgres reachable (server-enforced read-only session).")
    except Exception as error:  # noqa: BLE001
        print(f"Postgres unreachable ({type(error).__name__}) -> live-index cells skipped. Start it (python start.py) and re-run this cell.")


async def sql(query, **params):
    if not DB_OK:
        raise RuntimeError("Postgres not reachable: run `await check_db()` after starting it (python start.py).")
    async with sessions() as session:
        return (await session.execute(text(query), params)).mappings().all()

## 2. Offline structure (no DB)
Reads the cached Docling blocks in `data/ingest/<sha256>/`, runs the real `build_structure`, and gives you the draft
provisions/chunks exactly as ingestion would load them.

In [ ]:
import json as _json
from app.ingest.convert import read_blocks
from app.ingest.structure import build_structure, provision_path

manuals = {m["id"]: m for m in _json.loads((ROOT / "data" / "manuals.json").read_text(encoding="utf-8"))}
cache = {}
for status in (ROOT / "data" / "ingest").glob("*/status.json"):
    s = _json.loads(status.read_text(encoding="utf-8"))
    cache[s["document_id"]] = (status.parent, s["page_range"])
print({k: (v[0].name[:10], v[1]) for k, v in cache.items()})
print("manuals:", list(manuals))

In [ ]:
DOCUMENT = "fire"          # <- change me
FIRST_PAGE, LAST_PAGE = 21, 60   # pages 1-20 are cover/revision tables; None = through the end

out_dir, (p0, p1) = cache[DOCUMENT]
blocks = list(read_blocks(out_dir, FIRST_PAGE, LAST_PAGE or p1))
structure = build_structure(blocks)
print(f"{len(blocks)} blocks -> {len(structure.provisions)} provisions, {len(structure.chunks)} chunks")

In [ ]:
# Raw blocks (what Docling produced)
def show_block(i):
    b = blocks[i]
    print(f"[block {i}] kind={b.kind} page={b.page} method={b.method} conf={b.confidence} flags={b.flags}")
    print(wrap(b.text), "\n")

for i in range(5):
    show_block(i)

In [ ]:
# Provision tree (indent = depth)
def depth(i):
    d, cur = 0, structure.provisions[i].parent_index
    while cur is not None:
        d, cur = d + 1, structure.provisions[cur].parent_index
    return d

n_chunks = {}
for c in structure.chunks:
    n_chunks[c.provision_index] = n_chunks.get(c.provision_index, 0) + 1
for i, p in enumerate(structure.provisions[:60]):
    print(f"{i:4d} {'  ' * depth(i)}{p.full_identifier}  {p.title or ''}  ({n_chunks.get(i, 0)} chunks){' DUP' if p.duplicate else ''}")

In [ ]:
# Print ONE draft node: provision + its chunks + exact embedding input used at ingest
def draft_node(identifier=None, index=None, contextual=True):
    if index is None:
        matches = [i for i, p in enumerate(structure.provisions) if p.full_identifier == identifier]
        if not matches:
            raise LookupError(f"no provision {identifier!r}")
        index = matches[0]
    p = structure.provisions[index]
    parent = structure.provisions[p.parent_index].full_identifier if p.parent_index is not None else None
    kids = [q.full_identifier for q in structure.provisions if q.parent_index == index]
    print("=" * 100)
    print(f"PROVISION #{index}: {p.full_identifier} | title={p.title!r} | parent={parent} | children={kids} | sibling_order={p.sibling_order}")
    print("breadcrumb:", provision_path(structure.provisions, index))
    for c in (c for c in structure.chunks if c.provision_index == index):
        print("-" * 100)
        print(f"CHUNK ordinal={c.ordinal} pages={c.page_start}-{c.page_end} method={c.extraction_method} conf={c.extraction_confidence}")
        print(f"flags={c.flags} warnings={c.warnings}")
        print(f"table_markdown={'yes' if c.table_markdown else 'no'}  chars={len(c.raw_text)}")
        print(wrap(c.raw_text))
    return index

_ = draft_node(index=1)

## 3. Embedding sandbox (in memory)
Embed any node/text and see the vector, norms, similarities and a ranking over the draft chunks. Two input styles exist:
the baseline breadcrumb (`<id> <title> > ...\n<text>`) and the contextual `ctx1` form (`<manual title> > <id - title chain>\n<text>`),
which is what retrieval queries.

In [ ]:
def ctx1_input(chunk, label=None):
    label = label or manuals[DOCUMENT]["title"]
    parts, cur = [], chunk.provision_index
    while cur is not None:
        p = structure.provisions[cur]
        parts.append(f"{p.full_identifier} - {p.title}" if p.title else p.full_identifier)
        cur = p.parent_index
    return f"{label} > {' > '.join(reversed(parts))}\n{chunk.raw_text}"


def baseline_input(chunk):
    return f"{provision_path(structure.provisions, chunk.provision_index)}\n{chunk.raw_text}"


def embed_chunk(i, style="ctx1"):
    c = structure.chunks[i]
    t = ctx1_input(c) if style == "ctx1" else baseline_input(c)
    v = embedder.embed_documents([t])[0]
    print(f"chunk #{i} ({structure.provisions[c.provision_index].full_identifier}) style={style}")
    print("sha256(input):", hashlib.sha256(t.encode()).hexdigest())
    print("--- exact embedding input ---"); print(wrap(t, limit=800))
    print("--- vector ---"); show_vector(v)
    return v

v = embed_chunk(0)

In [ ]:
# Embed all draft chunks once (in memory), then query them
import time
t0 = time.time()
draft_inputs = [ctx1_input(c) for c in structure.chunks]
draft_vecs = embedder.embed_documents(draft_inputs)
print(f"embedded {len(draft_vecs)} chunks in {time.time() - t0:.1f}s")


def draft_search(query, k=5):
    q = embedder.embed_query(query)
    scored = sorted(((cosine(q, v), i) for i, v in enumerate(draft_vecs)), reverse=True)[:k]
    for rank, (s, i) in enumerate(scored, 1):
        c = structure.chunks[i]
        p = structure.provisions[c.provision_index]
        print(f"{rank}. cos={s:.4f} chunk#{i} {p.full_identifier} {p.title or ''} (p{c.page_start})")
        print("   ", wrap(c.raw_text, limit=220).replace("\n", " "))
    return scored

_ = draft_search("What is the minimum fire flow requirement?")

In [ ]:
# Pairwise similarity between two draft chunks / free text
def sim_chunks(i, j):
    print(f"cos(chunk#{i}, chunk#{j}) = {cosine(draft_vecs[i], draft_vecs[j]):.4f}")

def sim_texts(a, b):
    va, vb = embedder.embed_documents([a, b])
    print(f"cos = {cosine(va, vb):.4f}")

sim_chunks(0, 1)
sim_texts("fire hydrant spacing", "distance between hydrants")

## 4. Live index (Postgres, read-only)
What is actually stored: provisions, chunks, `chunk_embeddings` (model/revision/dims/hash/vector).
Contextual vectors are stored under revision `<embedder revision>#ctx1`.

In [ ]:
await check_db()
if DB_OK:
    rows = await sql("""
        SELECT d.id AS document, v.id AS version_id, v.source_title,
               (SELECT count(*) FROM provisions p WHERE p.source_version_id = v.id) AS provisions,
               (SELECT count(*) FROM chunks c WHERE c.source_version_id = v.id) AS chunks
        FROM documents d JOIN document_versions v ON v.id = d.current_source_version_id ORDER BY d.id
    """)
    for r in rows: print(dict(r))
    print()
    for r in await sql("SELECT model, revision, dimensions, count(*) AS n FROM chunk_embeddings GROUP BY 1,2,3 ORDER BY 4 DESC"):
        print(dict(r))


In [ ]:
VARIANT = "ctx1"
REVISION = f"{embedder.revision}#{VARIANT}"


async def db_node(document, identifier, vectors=True, head=8):
    """Print one stored provision, its parent/children, chunks and embedding rows."""
    prov = await sql("""
        SELECT p.id, p.full_identifier, p.title, p.sibling_order, p.parent_provision_id, p.source_version_id
        FROM provisions p JOIN documents d ON d.current_source_version_id = p.source_version_id
        WHERE d.id = :doc AND p.full_identifier = :ident
    """, doc=document, ident=identifier)
    if not prov:
        near = await sql("""SELECT p.full_identifier FROM provisions p JOIN documents d ON d.current_source_version_id = p.source_version_id
                            WHERE d.id = :doc AND p.full_identifier ILIKE :pat LIMIT 15""", doc=document, pat=f"%{identifier}%")
        print("not found; similar:", [r["full_identifier"] for r in near]); return
    p = prov[0]
    parent = (await sql("SELECT full_identifier, title FROM provisions WHERE id = :i", i=p["parent_provision_id"])) if p["parent_provision_id"] else []
    kids = await sql("SELECT full_identifier, title FROM provisions WHERE parent_provision_id = :i ORDER BY sibling_order", i=p["id"])
    print("=" * 100)
    print(f"PROVISION {p['full_identifier']} | title={p['title']!r} | sibling_order={p['sibling_order']} | id={p['id']}")
    print("parent  :", dict(parent[0]) if parent else None)
    print("children:", [f"{k['full_identifier']} {k['title'] or ''}".strip() for k in kids])
    chunks = await sql("SELECT * FROM chunks WHERE provision_id = :i ORDER BY ordinal", i=p["id"])
    for c in chunks:
        print("-" * 100)
        print(f"CHUNK ordinal={c['ordinal']} id={c['id']}")
        print(f"pages={c['physical_page_start']}-{c['physical_page_end']} printed={c['printed_page_label']} method={c['extraction_method']} conf={c['extraction_confidence']}")
        print(f"flags={c['quality_flags']} warnings={c['extraction_warnings']} table={'yes' if c['table_markdown'] else 'no'}")
        print(wrap(c["raw_text"]))
        embs = await sql("""SELECT model, revision, dimensions, embedding_input_sha256, created_at,
                                   embedding::text AS vec
                            FROM chunk_embeddings WHERE chunk_id = :i ORDER BY created_at""", i=c["id"])
        for e in embs:
            print(f"  EMBEDDING model={e['model']} rev={e['revision'][:12]}..{e['revision'][-6:]} dims={e['dimensions']} sha={e['embedding_input_sha256'][:12]} at={e['created_at']:%Y-%m-%d %H:%M}")
            if vectors:
                v = [float(x) for x in e["vec"].strip("[]").split(",")]
                print("   ", end=""); show_vector(v, head)
    return p

if DB_OK:
    _ = await db_node("fire", "1.2.0")

In [ ]:
# Find identifiers / text in the live index
async def find_nodes(document, pattern, limit=20):
    rows = await sql("""
        SELECT p.full_identifier, p.title, count(c.id) AS chunks
        FROM provisions p JOIN documents d ON d.current_source_version_id = p.source_version_id
        LEFT JOIN chunks c ON c.provision_id = p.id
        WHERE d.id = :doc AND (p.full_identifier ILIKE :pat OR p.title ILIKE :pat)
        GROUP BY p.id, p.sibling_order ORDER BY p.full_identifier LIMIT :lim
    """, doc=document, pat=f"%{pattern}%", lim=limit)
    for r in rows: print(f"{r['full_identifier']:<30} {r['title'] or '':<60} {r['chunks']} chunks")

if DB_OK:
    await find_nodes("fire", "hydrant")

In [ ]:
# Vector search against the live index (same SQL as app.rag.retrieval.VECTOR_SQL; no fusion/rerank)
async def live_search(query, k=8, documents=None, full_text=False):
    q = embedder.embed_query(query)
    vec = "[" + ",".join(repr(float(x)) for x in q) + "]"
    rows = await sql("""
        SELECT d.id AS document, p.full_identifier, p.title, c.id AS chunk_id, c.ordinal,
               c.physical_page_start AS page, left(c.raw_text, 240) AS snippet,
               1 - (e.embedding <=> CAST(:v AS vector)) AS cosine
        FROM chunk_embeddings e
        JOIN chunks c ON c.id = e.chunk_id
        JOIN provisions p ON p.id = c.provision_id
        JOIN documents d ON d.current_source_version_id = c.source_version_id
        WHERE (CAST(:docs AS text[]) IS NULL OR d.id = ANY(CAST(:docs AS text[])))
          AND e.model = :model AND e.revision = :rev AND e.dimensions = :dims
        ORDER BY e.embedding <=> CAST(:v AS vector), c.id LIMIT :k
    """, v=vec, docs=documents, model=embedder.model, rev=REVISION, dims=len(q), k=k)
    for rank, r in enumerate(rows, 1):
        print(f"{rank}. cos={r['cosine']:.4f} [{r['document']}] {r['full_identifier']} {r['title'] or ''} p{r['page']} ord={r['ordinal']}")
        print("   ", r["snippet"].replace("\n", " "))
    return rows

hits = await live_search("minimum fire flow for a commercial building") if DB_OK else []

In [ ]:
# Print any hit as a full node: run db_node on a rank from the search above
if DB_OK and hits:
    h = hits[0]
    await db_node(h["document"], h["full_identifier"])


In [ ]:
# Fetch a single chunk by id
async def db_chunk(chunk_id):
    r = (await sql("SELECT c.*, p.full_identifier, p.title FROM chunks c JOIN provisions p ON p.id = c.provision_id WHERE c.id = :i", i=chunk_id))[0]
    print(dict({k: (v if k != "raw_text" else wrap(v)) for k, v in r.items()}))

if DB_OK and hits:
    await db_chunk(hits[0]["chunk_id"])

In [ ]:
# Recompute the exact stored embedding input from DB rows and compare with its stored hash
from app.ingest.context import build_contextual_texts


async def verify_ctx1(document):
    r = await sql("SELECT v.id, v.source_title FROM documents d JOIN document_versions v ON v.id = d.current_source_version_id WHERE d.id = :d", d=document)
    vid, title = r[0]["id"], r[0]["source_title"]
    from types import SimpleNamespace as NS
    provs = [NS(**dict(x)) for x in await sql("SELECT id, full_identifier, title, parent_provision_id FROM provisions WHERE source_version_id = :v", v=vid)]
    chs = [NS(**dict(x)) for x in await sql("SELECT id, provision_id, ordinal, physical_page_start, raw_text FROM chunks WHERE source_version_id = :v", v=vid)]
    texts = build_contextual_texts(title, provs, chs, attach_unnumbered=False)
    stored = {x["chunk_id"]: x["h"] for x in await sql(
        "SELECT chunk_id, embedding_input_sha256 AS h FROM chunk_embeddings WHERE revision = :r AND model = :m", r=REVISION, m=embedder.model)}
    ok = sum(1 for cid, t in texts.items() if stored.get(cid) == hashlib.sha256(t.encode()).hexdigest())
    print(f"{document}: {ok}/{len(texts)} chunks have a ctx1 vector whose hash matches the recomputed input")

if DB_OK:
    await verify_ctx1("fire")

In [ ]:
if engine is not None:
    await engine.dispose()  # run last; re-run check_db to reconnect
